# Анализ графов Entity Resolution

Визуализация и анализ HeteroData-графов, построенных скриптом `02_build_graphs.py`.

Граф — двудольный (row ↔ token):
- **row**: строки обеих таблиц (A + B)
- **token**: уникальные субтокены ruBERT
- **edge_attr**: эмбеддинг столбца, через который токен связан со строкой

Загружаем из `data/graphs/<name>/` или `data/graphs/unified/`.

In [ ]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120
matplotlib.rcParams["figure.figsize"] = (12, 5)

GRAPHS_DIR = Path("../data/graphs")

# Загрузка сводки
summary_path = GRAPHS_DIR / "graph_summary.json"
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    print(f"Всего датасетов: {summary['total']}, failed: {summary['failed']}")
else:
    print("graph_summary.json не найден — сначала запустите 02_build_graphs.py --all")
    summary = None

## 1. Сводная статистика по всем графам

In [ ]:
# Собрать stats.json из каждого датасета
all_stats = []
for ds_dir in sorted(GRAPHS_DIR.iterdir()):
    stats_file = ds_dir / "stats.json"
    if ds_dir.is_dir() and ds_dir.name != "unified" and stats_file.exists():
        with open(stats_file) as f:
            all_stats.append(json.load(f))

df_stats = pd.DataFrame(all_stats)
cols = ["name", "domain", "n_rows", "n_rows_a", "n_rows_b",
        "n_tokens", "n_edges", "n_train_triplets", "n_val_triplets",
        "n_train_positives", "n_train_negatives"]
display(df_stats[cols].style
    .set_caption("Статистика per-dataset графов")
    .background_gradient(subset=["n_rows", "n_tokens", "n_edges"], cmap="YlOrRd")
    .format(thousands=",")
)

# Итого
print(f"\nВсего строк (row): {df_stats['n_rows'].sum():,}")
print(f"Всего токенов (token): {df_stats['n_tokens'].sum():,}")
print(f"Всего рёбер: {df_stats['n_edges'].sum():,}")
print(f"Всего train triplets: {df_stats['n_train_triplets'].sum():,}")
print(f"Всего val triplets: {df_stats['n_val_triplets'].sum():,}")

## 2. Распределение размеров графов по датасетам

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

names = df_stats["name"]
x = range(len(names))

# Узлы
axes[0].bar(x, df_stats["n_rows"], label="rows", alpha=0.8)
axes[0].bar(x, df_stats["n_tokens"], bottom=df_stats["n_rows"], label="tokens", alpha=0.8)
axes[0].set_title("Узлы (row + token)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=45, ha="right", fontsize=7)
axes[0].legend()
axes[0].set_ylabel("Количество")

# Рёбра
axes[1].bar(x, df_stats["n_edges"], color="C2", alpha=0.8)
axes[1].set_title("Рёбра (token ↔ row)")
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=45, ha="right", fontsize=7)
axes[1].set_ylabel("Количество")

# Триплеты
axes[2].bar(x, df_stats["n_train_triplets"], label="train", alpha=0.8)
axes[2].bar(x, df_stats["n_val_triplets"], bottom=df_stats["n_train_triplets"],
            label="val", alpha=0.8)
axes[2].set_title("Триплеты (train + val)")
axes[2].set_xticks(x)
axes[2].set_xticklabels(names, rotation=45, ha="right", fontsize=7)
axes[2].legend()
axes[2].set_ylabel("Количество")

plt.tight_layout()
plt.show()

## 3. Загрузка и инспекция одного графа

In [ ]:
# Выбрать датасет для детального анализа
DATASET_NAME = df_stats.iloc[0]["name"]  # можно заменить на любой
print(f"Детальный анализ: {DATASET_NAME}\n")

graph = torch.load(GRAPHS_DIR / DATASET_NAME / "graph.pt", weights_only=False)
train_tri = torch.load(GRAPHS_DIR / DATASET_NAME / "train_triplets.pt", weights_only=False)
val_tri = torch.load(GRAPHS_DIR / DATASET_NAME / "val_triplets.pt", weights_only=False)

print(f"row.x:    {graph['row'].x.shape}")
print(f"token.x:  {graph['token'].x.shape}")
print(f"edge (token→row): {graph['token', 'in_row', 'row'].edge_index.shape}")
print(f"edge_attr:         {graph['token', 'in_row', 'row'].edge_attr.shape}")
print(f"train triplets:    {train_tri.shape}")
print(f"val triplets:      {val_tri.shape}")

## 4. Распределение степеней вершин (degree distribution)

In [ ]:
ei = graph["token", "in_row", "row"].edge_index  # [2, E]

# Степень row-узлов (сколько токенов связано с каждой строкой)
row_deg = torch.zeros(graph["row"].x.shape[0], dtype=torch.long)
row_deg.scatter_add_(0, ei[1], torch.ones(ei.shape[1], dtype=torch.long))

# Степень token-узлов (в скольких строках встречается токен)
tok_deg = torch.zeros(graph["token"].x.shape[0], dtype=torch.long)
tok_deg.scatter_add_(0, ei[0], torch.ones(ei.shape[1], dtype=torch.long))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(row_deg.numpy(), bins=50, color="C0", alpha=0.8, edgecolor="white")
axes[0].set_title(f"Степень row-узлов ({DATASET_NAME})")
axes[0].set_xlabel("Степень (кол-во связанных токенов)")
axes[0].set_ylabel("Кол-во строк")
axes[0].axvline(row_deg.float().mean(), color="red", ls="--", label=f"mean={row_deg.float().mean():.1f}")
axes[0].legend()

axes[1].hist(tok_deg.numpy(), bins=50, color="C1", alpha=0.8, edgecolor="white")
axes[1].set_title(f"Степень token-узлов ({DATASET_NAME})")
axes[1].set_xlabel("Степень (кол-во строк с этим токеном)")
axes[1].set_ylabel("Кол-во токенов")
axes[1].axvline(tok_deg.float().mean(), color="red", ls="--", label=f"mean={tok_deg.float().mean():.1f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Row degree: min={row_deg.min()}, max={row_deg.max()}, "
      f"mean={row_deg.float().mean():.1f}, median={row_deg.float().median():.0f}")
print(f"Token degree: min={tok_deg.min()}, max={tok_deg.max()}, "
      f"mean={tok_deg.float().mean():.1f}, median={tok_deg.float().median():.0f}")

## 5. Log-log график степени token-узлов

Проверка scale-free свойства: если линейно в log-log — степенной закон.

In [ ]:
deg_counts = Counter(tok_deg.numpy().tolist())
degs = sorted(deg_counts.keys())
freqs = [deg_counts[d] for d in degs]

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(degs, freqs, s=10, alpha=0.7)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Степень токена (log)")
ax.set_ylabel("Частота (log)")
ax.set_title(f"Log-log степенное распределение токенов ({DATASET_NAME})")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Соотношение row/token по датасетам и плотность графа

In [ ]:
# Плотность двудольного графа: E / (N_row * N_token)
df_stats["density"] = df_stats["n_edges"] / (df_stats["n_rows"] * df_stats["n_tokens"])
df_stats["token_per_row"] = df_stats["n_edges"] / df_stats["n_rows"]
df_stats["rows_per_token"] = df_stats["n_edges"] / df_stats["n_tokens"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].barh(df_stats["name"], df_stats["density"], color="C3", alpha=0.8)
axes[0].set_title("Плотность графа (E / N_row×N_token)")
axes[0].set_xlabel("Плотность")

axes[1].barh(df_stats["name"], df_stats["token_per_row"], color="C0", alpha=0.8)
axes[1].set_title("Среднее рёбер на строку")
axes[1].set_xlabel("E / N_row")

axes[2].barh(df_stats["name"], df_stats["rows_per_token"], color="C1", alpha=0.8)
axes[2].set_title("Среднее строк на токен")
axes[2].set_xlabel("E / N_token")

plt.tight_layout()
plt.show()

## 7. Анализ связности: общие токены между таблицами A и B

Ключевой вопрос: насколько таблицы A и B связаны через общие токены?
Чем больше общих токенов — тем лучше GNN может передавать информацию между строками.

In [ ]:
with open(GRAPHS_DIR / DATASET_NAME / "id_to_global_a.json") as f:
    id_to_global_a = json.load(f)
with open(GRAPHS_DIR / DATASET_NAME / "id_to_global_b.json") as f:
    id_to_global_b = json.load(f)

rows_a = set(id_to_global_a.values())
rows_b = set(id_to_global_b.values())

# Токены, связанные с A и B
tok_src = ei[0].numpy()
row_dst = ei[1].numpy()

tokens_of_a = set(tok_src[np.isin(row_dst, list(rows_a))])
tokens_of_b = set(tok_src[np.isin(row_dst, list(rows_b))])
shared_tokens = tokens_of_a & tokens_of_b

n_tok = graph["token"].x.shape[0]
print(f"Токены только в A:  {len(tokens_of_a - tokens_of_b)} / {n_tok} ({100*len(tokens_of_a - tokens_of_b)/n_tok:.1f}%)")
print(f"Токены только в B:  {len(tokens_of_b - tokens_of_a)} / {n_tok} ({100*len(tokens_of_b - tokens_of_a)/n_tok:.1f}%)")
print(f"Общие токены (A∩B): {len(shared_tokens)} / {n_tok} ({100*len(shared_tokens)/n_tok:.1f}%)")
print(f"\n→ Через {len(shared_tokens)} общих токенов GNN связывает строки таблиц A и B")

## 8. t-SNE визуализация row-эмбеддингов (A vs B)

In [ ]:
from sklearn.manifold import TSNE

row_x = graph["row"].x.numpy()
n_a = len(rows_a)

# Сэмплирование если слишком много строк
MAX_SAMPLE = 2000
if row_x.shape[0] > MAX_SAMPLE:
    idx = np.random.choice(row_x.shape[0], MAX_SAMPLE, replace=False)
    idx = np.sort(idx)
    row_x_sample = row_x[idx]
    labels = ["A" if i < n_a else "B" for i in idx]
else:
    row_x_sample = row_x
    labels = ["A"] * n_a + ["B"] * (row_x.shape[0] - n_a)

perplexity = min(30, len(row_x_sample) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
coords = tsne.fit_transform(row_x_sample)

fig, ax = plt.subplots(figsize=(8, 6))
for label, color in [("A", "C0"), ("B", "C1")]:
    mask = [l == label for l in labels]
    ax.scatter(coords[mask, 0], coords[mask, 1], s=5, alpha=0.5,
               label=f"Table {label}", color=color)
ax.set_title(f"t-SNE row-эмбеддингов ({DATASET_NAME})")
ax.legend()
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

## 9. Unified граф: статистика

In [ ]:
unified_stats_path = GRAPHS_DIR / "unified" / "stats.json"
if unified_stats_path.exists():
    with open(unified_stats_path) as f:
        u_stats = json.load(f)
    
    print("=" * 50)
    print("UNIFIED GRAPH")
    print("=" * 50)
    print(f"Датасетов:      {u_stats['n_datasets']}")
    print(f"Rows (total):   {u_stats['n_rows_total']:,}")
    print(f"Tokens (total): {u_stats['n_tokens_total']:,}")
    print(f"Edges (total):  {u_stats['n_edges_total']:,}")
    print(f"Train triplets: {u_stats['n_train_triplets']:,}")
    print(f"Val triplets:   {u_stats['n_val_triplets']:,}")
    
    if u_stats.get("failed"):
        print(f"\nFailed: {u_stats['failed']}")
    
    # Pie chart: вклад каждого датасета в rows
    ds_list = u_stats["datasets"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    ds_names = [d["name"] for d in ds_list]
    ds_rows = [d["n_rows"] for d in ds_list]
    axes[0].pie(ds_rows, labels=ds_names, autopct="%1.0f%%", textprops={"fontsize": 7})
    axes[0].set_title("Доля строк по датасетам (unified)")
    
    # Bar: row_offset — показывает расположение каждого датасета в графе
    offsets = [d["row_offset"] for d in ds_list]
    sizes = [d["n_rows"] for d in ds_list]
    axes[1].barh(ds_names, sizes, left=offsets, alpha=0.8)
    axes[1].set_xlabel("Глобальный индекс строк")
    axes[1].set_title("Расположение датасетов в unified графе")
    
    plt.tight_layout()
    plt.show()
else:
    print("Unified граф не построен — запустите 02_build_graphs.py --all")

## 10. Сравнение средней степени по доменам

In [ ]:
domain_agg = df_stats.groupby("domain").agg({
    "n_rows": "mean",
    "n_tokens": "mean",
    "n_edges": "mean",
    "density": "mean",
    "token_per_row": "mean",
    "name": "count",
}).rename(columns={"name": "n_datasets"}).sort_values("n_edges", ascending=False)

display(domain_agg.style
    .set_caption("Средние метрики по доменам")
    .format({"n_rows": "{:.0f}", "n_tokens": "{:.0f}", "n_edges": "{:.0f}",
             "density": "{:.4f}", "token_per_row": "{:.1f}"})
    .background_gradient(cmap="Blues")
)